# 🩺 SonoVision AI: Vision Transformer (ViT-B/16) on BUSI Dataset
### M.Tech AI/ML Project — Breast Ultrasound Classification & Grad-CAM Explainability
---
**Architecture:** Pretrained Vision Transformer (`ViT-B/16`)
**Dataset:** Breast Ultrasound Images (BUSI) [Benign, Malignant, Normal]
**Explainability:** Patch-Token Gradient-Weighted Class Activation Mapping (Grad-CAM)
**Hardware:** Google Colab GPU (T4 / V100 / A100) or CPU


## 1. Environment Setup & Hardware Verification
Verify CUDA GPU availability and install required libraries.

In [ ]:
!nvidia-smi
!pip install -q torch torchvision timm opencv-python matplotlib seaborn scikit-learn pandas pillow tqdm

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, auc, classification_report
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
import cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Compute Device: {device}')

## 2. Dataset Acquisition & Leakage Prevention
The BUSI dataset contains 780 scans with segmentation masks (e.g. `benign (1).png` alongside `benign (1)_mask.png`).
**Crucial Data Hygiene:** We filter out all `*_mask.png` files to ensure only real ultrasound scans are fed into the classifier, preventing feature leakage.

In [ ]:
# Download BUSI dataset from public mirror or Kaggle if not present
DATA_DIR = Path('./Dataset_BUSI_with_GT')
if not DATA_DIR.exists():
    print('Downloading BUSI Dataset...')
    !wget -q -O busi.zip 'https://scholar.cu.edu.eg/dataset/busi.zip' || \
    curl -L -o busi.zip 'https://www.kaggle.com/api/v1/datasets/download/aryashah2k/breast-ultrasound-images-dataset'
    !unzip -q busi.zip -d ./Dataset_BUSI_with_GT || true
    if (DATA_DIR / 'Dataset_BUSI_with_GT').exists():
        DATA_DIR = DATA_DIR / 'Dataset_BUSI_with_GT'
print(f'Target BUSI folder: {DATA_DIR}')

In [ ]:
# Index dataset avoiding mask leakage
classes = ['benign', 'malignant', 'normal']
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

image_paths = []
labels = []

for cls in classes:
    cls_dir = DATA_DIR / cls
    if not cls_dir.exists():
        for d in DATA_DIR.iterdir():
            if d.is_dir() and d.name.lower() == cls:
                cls_dir = d
                break
    for img in sorted(cls_dir.glob('*.png')):
        if '_mask' in img.name.lower():
            continue  # Exclude masks!
        image_paths.append(img)
        labels.append(class_to_idx[cls])

print(f'Total valid ultrasound scans: {len(image_paths)}')
for cls in classes:
    print(f'  - {cls}: {labels.count(class_to_idx[cls])}')

## 3. Stratified Train / Validation / Test Splitting
We use a strict 70% / 10% / 20% stratified split to guarantee zero data leakage between sets while preserving class proportions.

In [ ]:
train_paths, test_paths, train_lbls, test_lbls = train_test_split(
    image_paths, labels, test_size=0.20, stratify=labels, random_state=42
)
train_paths, val_paths, train_lbls, val_lbls = train_test_split(
    train_paths, train_lbls, test_size=0.125, stratify=train_lbls, random_state=42
)
print(f'Split Sizes: Train={len(train_paths)}, Val={len(val_paths)}, Test={len(test_paths)}')

## 4. PyTorch Dataset & Pretrained ImageNet Normalization
Vision Transformer requires 224x224 images normalized with ImageNet mean `[0.485, 0.456, 0.406]` and std `[0.229, 0.224, 0.225]`.

In [ ]:
class BUSIDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.file_paths)
    def __getitem__(self, idx):
        img = Image.open(self.file_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(BUSIDataset(train_paths, train_lbls, train_tfms), batch_size=16, shuffle=True)
val_loader = DataLoader(BUSIDataset(val_paths, val_lbls, eval_tfms), batch_size=16, shuffle=False)
test_loader = DataLoader(BUSIDataset(test_paths, test_lbls, eval_tfms), batch_size=16, shuffle=False)

## 5. Vision Transformer Architecture (ViT-B/16)
We adapt the standard ViT-B/16 classifier head (768 -> 3) and calculate class weights to mitigate the class imbalance (Benign > Malignant > Normal).

In [ ]:
model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
model.heads.head = nn.Linear(model.heads.head.in_features, len(classes))
model = model.to(device)

class_counts = [labels.count(i) for i in range(len(classes))]
class_weights = [len(labels) / (len(classes) * c) for c in class_counts]
class_weights_t = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f'Inverse Class Weights: {class_weights}')

criterion = nn.CrossEntropyLoss(weight=class_weights_t, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-6)

## 6. Training with Early Stopping & Validation Checkpointing

In [ ]:
EPOCHS = 25
PATIENCE = 7
best_loss = float('inf')
patience_cnt = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, t_corr, t_tot = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_corr += (out.argmax(1) == lbls).sum().item()
        t_tot += lbls.size(0)
    scheduler.step()
    
    model.eval()
    v_loss, v_corr, v_tot = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            loss = criterion(out, lbls)
            v_loss += loss.item() * imgs.size(0)
            v_corr += (out.argmax(1) == lbls).sum().item()
            v_tot += lbls.size(0)
    
    tl, ta = t_loss / t_tot, t_corr / t_tot
    vl, va = v_loss / v_tot, v_corr / v_tot
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    
    print(f'Epoch {epoch:02d} | Train Loss: {tl:.4f} Acc: {ta:.4f} | Val Loss: {vl:.4f} Acc: {va:.4f}')
    
    if vl < best_loss:
        best_loss = vl
        patience_cnt = 0
        torch.save(model.state_dict(), 'best_vit_b16_busi.pth')
        print(f'  --> Best checkpoint saved (Val Loss: {vl:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print('Early stopping triggered!')
            break

## 7. Held-Out Test Evaluation & Publication Metrics
Calculate Accuracy, Precision, Recall, F1, Per-class Sensitivity, Specificity, Confusion Matrix, and ROC Curves.

In [ ]:
model.load_state_dict(torch.load('best_vit_b16_busi.pth', map_location=device))
model.eval()
all_preds, all_lbls, all_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        all_probs.extend(F.softmax(out, dim=1).cpu().numpy())
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_lbls.extend(lbls.numpy())

all_lbls = np.array(all_lbls)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

print(classification_report(all_lbls, all_preds, target_names=classes))

# Confusion Matrix Plot
cm = confusion_matrix(all_lbls, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Held-Out Test Set Confusion Matrix (ViT-B/16)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# ROC Curve Plot
plt.figure(figsize=(7, 6))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(all_lbls == i, all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{cls.capitalize()} (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.title('Multi-Class ROC Curves (Held-Out Test Set)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Vision Transformer Patch-Token Grad-CAM Explainability
Hooks into the final Transformer encoder layer `model.encoder.layers[-1].ln_1`, extracts gradients for the 196 patch tokens, reshapes to $14 \times 14$, and overlays on ultrasound images.

In [ ]:
class ViTGradCAMExplainer:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.acts, self.grads = None, None
        self.target_layer.register_forward_hook(lambda m, i, o: setattr(self, 'acts', o.detach()))
        self.target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'grads', go[0].detach()))
    def generate(self, img_tensor, cls_idx=None):
        self.model.eval()
        img_tensor = img_tensor.clone().requires_grad_(True)
        out = self.model(img_tensor)
        if cls_idx is None: cls_idx = out.argmax().item()
        out[0, cls_idx].backward()
        patch_grads = self.grads[:, 1:, :]
        patch_acts = self.acts[:, 1:, :]
        weights = patch_grads.mean(dim=1, keepdim=True)
        cam = F.relu((patch_acts * weights).sum(dim=-1)).view(1, 1, 14, 14)
        cam_up = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False).squeeze().cpu().numpy()
        cam_norm = (cam_up - cam_up.min()) / (cam_up.max() - cam_up.min() + 1e-8)
        return cam_norm, cls_idx

explainer = ViTGradCAMExplainer(model, model.encoder.layers[-1].ln_1)
print('ViT-B/16 Grad-CAM Explainer initialized.')

## 9. Visualizing Sample Ultrasound Scans with Heatmaps

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
sample_indices = [0, len(test_paths)//2, len(test_paths)-1]

for row, idx in enumerate(sample_indices):
    img_tensor, true_label = test_ds[idx]
    cam, pred_label = explainer.generate(img_tensor.unsqueeze(0).to(device))
    
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = 0.5 * heatmap + 0.5 * img_np
    
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f'Original: {classes[true_label]}')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(cam, cmap='jet')
    axes[row, 1].set_title('ViT Patch Grad-CAM Heatmap')
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title(f'Overlay | Pred: {classes[pred_label]}')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()